In [ ]:
!pip install transformers sentencepiece sacremoses pandas openpyxl torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 18.5 MB/s eta 0:00:00


In [ ]:
# ============================================
# ENGLISH TO ASSAMESE & MANIPURI TRANSLATION
# Using NLLB-200 Model (FIXED)
# ============================================

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION
# ============================================

# Output directory
OUTPUT_DIR = "/content/drive/MyDrive/wmt26nllb/"

# NLLB-200 Model
MODEL_NAME = "facebook/nllb-200-distilled-600M"

# Language codes for NLLB-200 (using FLORES-200 codes)
SRC_LANG = "eng_Latn"  # English

# Translation tasks with correct NLLB language codes
TRANSLATION_TASKS = [
    {
        "input_file": "/content/en-as Test.xlsx",
        "target_lang": "asm_Beng",  # Assamese in Bengali script
        "target_name": "Assamese",
        "output_name": "english_to_assamese",
        "column_name": "Assamese_Sentence"
    },
    {
        "input_file": "/content/en-mni Test.xlsx",
        "target_lang": "mni_Beng",  # Manipuri in Bengali script
        "target_name": "Manipuri",
        "output_name": "english_to_manipuri",
        "column_name": "Manipuri_Bengali_Sentence"
    }
]

# Translation settings
BATCH_SIZE = 2  # Small batch size for CPU
MAX_LENGTH = 512
NUM_BEAMS = 5
REPETITION_PENALTY = 1.2

# ============================================
# MOUNT GOOGLE DRIVE
# ============================================
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

# ============================================
# CHECK GPU
# ============================================
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("⚠️  Using CPU - translations will be slower")
    print("   Recommend: Runtime → Change runtime type → T4 GPU")

# ============================================
# LOAD NLLB MODEL
# ============================================
print(f"\n🔄 Loading NLLB-200 model...")
print(f"   Model: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(DEVICE)

model.eval()
print("✅ NLLB model loaded successfully!\n")

# ============================================
# TRANSLATION FUNCTION FOR NLLB (CORRECTED)
# ============================================
def translate_batch_nllb(sentences, target_lang, target_name):
    """
    Translate English sentences using NLLB model
    The key is to set the source and target languages in the tokenizer
    """
    translations = []
    total = len(sentences)
    total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"\n{'='*80}")
    print(f"🎯 TRANSLATING ENGLISH TO {target_name.upper()}")
    print(f"{'='*80}")
    print(f"Total sentences: {total}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Total batches: {total_batches}")
    print(f"Target code: {target_lang}")
    print(f"{'='*80}\n")

    start_time = time.time()

    # Set the source language in tokenizer
    tokenizer.src_lang = SRC_LANG

    for batch_idx in range(0, total, BATCH_SIZE):
        batch = sentences[batch_idx:batch_idx + BATCH_SIZE]
        batch_num = batch_idx // BATCH_SIZE + 1

        print(f"\n{'─'*80}")
        print(f"📦 BATCH {batch_num}/{total_batches}")
        print(f"{'─'*80}")

        # Tokenize inputs with source language
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(DEVICE)

        # Generate translations - IMPORTANT: Use generate with target language
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
                max_length=MAX_LENGTH,
                num_beams=NUM_BEAMS,
                repetition_penalty=REPETITION_PENALTY,
                early_stopping=True,
            )

        # Decode translations
        batch_translations = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        translations.extend(batch_translations)

        # Display batch results
        for i, (source, target) in enumerate(zip(batch, batch_translations)):
            sent_num = batch_idx + i + 1
            print(f"\n   📝 SENTENCE {sent_num}/{total}")
            print(f"   🇬🇧 English: {source[:80]}{'...' if len(source) > 80 else ''}")
            print(f"   🪶 {target_name}: {target[:80]}{'...' if len(target) > 80 else ''}")

            # Check if translation is different from source
            if source.strip() == target.strip():
                print(f"   ⚠️  WARNING: Translation appears to be copying English!")
            else:
                print(f"   ✅ Translation seems successful!")

            print(f"   📊 EN length: {len(source)} chars | {target_name} length: {len(target)} chars")

        # Show progress
        elapsed = time.time() - start_time
        avg_time_per_batch = elapsed / batch_num
        remaining_batches = total_batches - batch_num
        eta = avg_time_per_batch * remaining_batches

        print(f"\n   ⏱️  Progress: {min(batch_idx + BATCH_SIZE, total)}/{total} ({min(batch_idx + BATCH_SIZE, total)/total*100:.1f}%)")
        print(f"   ⏱️  ETA: {eta/60:.1f} minutes | Elapsed: {elapsed/60:.1f} minutes")

        # Clear GPU cache
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    end_time = time.time()
    total_time = end_time - start_time

    print(f"\n{'='*80}")
    print(f"✅ COMPLETED: English to {target_name}")
    print(f"{'='*80}")
    print(f"⏱️  Total time: {total_time/60:.2f} minutes")
    print(f"📊 Average: {total_time/total:.2f} seconds/sentence")
    print(f"📊 Speed: {total/(total_time/60):.1f} sentences/minute")

    return translations, total_time

def process_file_nllb(task):
    """
    Process a single translation task
    """
    print("\n" + "="*80)
    print(f"📁 LOADING FILE: {task['target_name']}")
    print("="*80)
    print(f"Input file: {task['input_file']}")
    print(f"Target language: {task['target_lang']}")

    # Check if input file exists
    if not os.path.exists(task['input_file']):
        print(f"❌ File not found: {task['input_file']}")
        print("\n📁 Available files in /content:")
        for f in os.listdir('/content'):
            print(f"   - {f}")
        return None

    # Read Excel file
    try:
        df = pd.read_excel(task['input_file'], engine='openpyxl')
        print(f"✅ Excel loaded successfully!")
        print(f"📊 Columns: {df.columns.tolist()}")
        print(f"📊 Total rows: {len(df)}")
    except Exception as e:
        print(f"❌ Error reading Excel: {e}")
        return None

    # Find English sentences column
    english_col = None
    for col in df.columns:
        if 'english' in col.lower() or 'target' in col.lower() or 'sentence' in col.lower():
            english_col = col
            break

    if english_col is None:
        english_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]

    print(f"✅ Using column: '{english_col}'")

    # Extract and clean sentences
    english_sentences = df[english_col].dropna().astype(str).tolist()
    english_sentences = [s.strip() for s in english_sentences if len(s.strip()) > 3 and not s.strip().isdigit()]

    print(f"✅ Found {len(english_sentences)} valid English sentences")

    if len(english_sentences) == 0:
        print("❌ No valid sentences found!")
        return None

    # Preview first few sentences
    print("\n📝 First 5 English sentences:")
    for i, sent in enumerate(english_sentences[:5]):
        print(f"   {i+1}. {sent[:100]}{'...' if len(sent) > 100 else ''}")

    # Translate
    translations, total_time = translate_batch_nllb(
        english_sentences,
        task['target_lang'],
        task['target_name']
    )

    # Save results
    save_results_nllb(english_sentences, translations, task, total_time)

    return {
        'sentences': english_sentences,
        'translations': translations,
        'time': total_time,
        'successful': sum(1 for t in translations if t and t != english_sentences[i] for i in range(len(translations)))
    }

def save_results_nllb(english_sentences, translations, task, total_time):
    """
    Save translation results in .txt format
    """
    print(f"\n💾 Saving {task['target_name']} results...")

    # Count successful translations (not copying English)
    successful = sum(1 for eng, trans in zip(english_sentences, translations) if trans and trans.strip() != eng.strip())

    # 1. Formatted text file with English and translation
    txt_path = os.path.join(OUTPUT_DIR, f"{task['output_name']}.txt")
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write(f"ENGLISH TO {task['target_name'].upper()} TRANSLATIONS\n")
        f.write("="*80 + "\n")
        f.write(f"Model: {MODEL_NAME}\n")
        f.write(f"Source file: {task['input_file']}\n")
        f.write(f"Language pair: {SRC_LANG} → {task['target_lang']}\n")
        f.write(f"Total sentences: {len(english_sentences)}\n")
        f.write(f"Successful translations: {successful}/{len(translations)}\n")
        f.write(f"Translation time: {total_time/60:.2f} minutes\n")
        f.write(f"Average speed: {len(english_sentences)/(total_time/60):.1f} sentences/minute\n")
        f.write("="*80 + "\n\n")

        for idx, (eng, trans) in enumerate(zip(english_sentences, translations), 1):
            f.write(f"[{idx}] ENGLISH:\n")
            f.write(f"{eng}\n\n")
            f.write(f"[{idx}] {task['target_name']} ({task['target_lang']}):\n")
            if trans and trans != eng:
                f.write(f"{trans}\n")
            else:
                f.write(f"[TRANSLATION FAILED - Output: {trans}]\n")
            f.write("-"*80 + "\n\n")

    print(f"   ✅ Formatted TXT: {txt_path}")

    # 2. Target language only text file
    target_only_path = os.path.join(OUTPUT_DIR, f"{task['output_name']}_target_only.txt")
    with open(target_only_path, 'w', encoding='utf-8') as f:
        for eng, trans in zip(english_sentences, translations):
            if trans and trans != eng:
                f.write(trans + '\n')
            else:
                f.write('' + '\n')

    print(f"   ✅ Target-only TXT: {target_only_path}")

    # 3. English and translation side-by-side text file
    side_by_side_path = os.path.join(OUTPUT_DIR, f"{task['output_name']}_side_by_side.txt")
    with open(side_by_side_path, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write(f"ENGLISH ↔ {task['target_name'].upper()} (Side by Side)\n")
        f.write("="*80 + "\n\n")

        for idx, (eng, trans) in enumerate(zip(english_sentences, translations), 1):
            f.write(f"[{idx}]\n")
            f.write(f"EN: {eng}\n")
            if trans and trans != eng:
                f.write(f"{task['target_name']}: {trans}\n")
            else:
                f.write(f"{task['target_name']}: [TRANSLATION FAILED]\n")
            f.write("-"*80 + "\n")

    print(f"   ✅ Side-by-side TXT: {side_by_side_path}")

    # 4. CSV file for Excel compatibility
    csv_path = os.path.join(OUTPUT_DIR, f"{task['output_name']}.csv")
    df_out = pd.DataFrame({
        'ID': range(1, len(english_sentences) + 1),
        'English_Sentence': english_sentences,
        task['column_name']: translations,
        'Translation_Successful': [1 if (trans and trans != eng) else 0 for eng, trans in zip(english_sentences, translations)],
        'English_Length': [len(s) for s in english_sentences],
        'Translation_Length': [len(t) for t in translations]
    })
    df_out.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"   ✅ CSV: {csv_path}")

# ============================================
# MAIN EXECUTION
# ============================================
print("\n" + "="*80)
print("🎯 NLLB-200 TRANSLATION SYSTEM")
print("="*80)
print(f"Model: {MODEL_NAME}")
print(f"Source language: {SRC_LANG} (English)")
print(f"Target languages: Assamese (asm_Beng), Manipuri (mni_Beng)")
print(f"Output directory: {OUTPUT_DIR}")
print("="*80)

all_results = {}
total_start_time = time.time()

# Test with a single sentence first to verify translation works
print("\n🔍 Testing translation with a sample sentence...")
test_sentence = "Hello, how are you?"
print(f"Test English: {test_sentence}")

tokenizer.src_lang = SRC_LANG
test_inputs = tokenizer([test_sentence], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
test_outputs = model.generate(
    **test_inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("asm_Beng"),
    max_length=50,
    num_beams=5
)
test_translation = tokenizer.batch_decode(test_outputs, skip_special_tokens=True)[0]
print(f"Test Assamese: {test_translation}")
print(f"Test successful: {'✅' if test_translation != test_sentence else '❌ Not translating properly'}")
print()

for i, task in enumerate(TRANSLATION_TASKS, 1):
    print(f"\n{'#'*80}")
    print(f"TASK {i}/{len(TRANSLATION_TASKS)}")
    print(f"{'#'*80}")

    result = process_file_nllb(task)
    if result:
        all_results[task['target_name']] = result

    # Clear cache between files
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # Small pause between files
    if i < len(TRANSLATION_TASKS):
        print("\n" + "="*80)
        print("Moving to next language...")
        time.sleep(2)

total_end_time = time.time()
total_elapsed = total_end_time - total_start_time

# ============================================
# FINAL SUMMARY
# ============================================
print("\n" + "="*80)
print("🎉 ALL TRANSLATIONS COMPLETE!")
print("="*80)
print(f"⏱️  Total time: {total_elapsed/60:.2f} minutes")

print("\n📊 SUMMARY BY LANGUAGE:")
print("-"*80)
for lang_name, result in all_results.items():
    successful = result['successful']
    total = len(result['sentences'])
    print(f"\n{lang_name}:")
    print(f"   • Sentences: {total}")
    print(f"   • Successful: {successful}/{total} ({successful/total*100:.1f}%)")
    print(f"   • Time: {result['time']/60:.2f} minutes")
    print(f"   • Speed: {total/(result['time']/60):.1f} sentences/minute")

print("\n📁 All files saved to:")
print(f"   {OUTPUT_DIR}")

print("\n📄 Output files per language:")
for task in TRANSLATION_TASKS:
    print(f"\n   {task['target_name']}:")
    print(f"      • {task['output_name']}.txt (Formatted with English)")
    print(f"      • {task['output_name']}_target_only.txt (Translations only)")
    print(f"      • {task['output_name']}_side_by_side.txt (Side by side)")
    print(f"      • {task['output_name']}.csv (Excel format)")

# ============================================
# CREATE MASTER SUMMARY FILE
# ============================================
print("\n" + "="*80)
print("📊 CREATING MASTER SUMMARY")
print("="*80)

master_summary_path = os.path.join(OUTPUT_DIR, "translation_summary.txt")
with open(master_summary_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("NLLB-200 TRANSLATION SUMMARY\n")
    f.write("="*80 + "\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write(f"Source: English ({SRC_LANG})\n")
    f.write(f"Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total processing time: {total_elapsed/60:.2f} minutes\n")
    f.write("="*80 + "\n\n")

    for lang_name, result in all_results.items():
        f.write(f"\n{lang_name}:\n")
        f.write(f"   • Sentences: {len(result['sentences'])}\n")
        f.write(f"   • Successful: {result['successful']}/{len(result['translations'])}\n")
        f.write(f"   • Time: {result['time']/60:.2f} minutes\n")
        f.write(f"   • Speed: {len(result['sentences'])/(result['time']/60):.1f} sentences/minute\n")

print(f"✅ Master summary: {master_summary_path}")

print("\n" + "="*80)
print("🎉 PROCESS COMPLETE!")
print("="*80)
print("\n✅ You can now download all translation files from Google Drive.")
print(f"   Location: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Output directory: /content/drive/MyDrive/wmt26nllb/
⚠️  Using CPU - translations will be slower
   Recommend: Runtime → Change runtime type → T4 GPU

🔄 Loading NLLB-200 model...
   Model: facebook/nllb-200-distilled-600M


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ NLLB model loaded successfully!


🎯 NLLB-200 TRANSLATION SYSTEM
Model: facebook/nllb-200-distilled-600M
Source language: eng_Latn (English)
Target languages: Assamese (asm_Beng), Manipuri (mni_Beng)
Output directory: /content/drive/MyDrive/wmt26nllb/

🔍 Testing translation with a sample sentence...
Test English: Hello, how are you?


In [ ]:
import os
import re


def split_translation_file(input_file, output_dir, target_lang_name):
    """
    Split a translation file containing both Original and Translated lines
    into separate English-only and Target-language-only files.

    Args:
        input_file: Path to the input file with Original and Translated lines
        output_dir: Directory to save the output files
        target_lang_name: Name of the target language (for file naming)
    """

    # Check if input file exists
    if not os.path.exists(input_file):
        print(f"❌ File not found: {input_file}")
        return None, None

    # Read the file
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    print(f"\n📁 Processing: {os.path.basename(input_file)}")
    print(f"   Total lines in file: {len(lines)}")

    # Extract English and Target language sentences
    english_sentences = []
    target_sentences = []

    for line in lines:
        line = line.strip()
        if not line:  # Skip empty lines
            continue

        # Check if it's an Original or Translated line
        if line.startswith('Original'):
            # Extract the English sentence after the number
            # Format: "Original (1): NDTV has learnt..."
            match = re.search(r'Original\s*\(\d+\):\s*(.*)', line)
            if match:
                english_sentences.append(match.group(1).strip())
        elif line.startswith('Translated'):
            # Extract the translated sentence
            # Format: "Translated: এনডিটিভিয়ে সূত্ৰৰ পৰা জানিব..."
            match = re.search(r'Translated:\s*(.*)', line)
            if match:
                target_sentences.append(match.group(1).strip())

    # Verify we have matching counts
    print(f"   English sentences found: {len(english_sentences)}")
    print(f"   {target_lang_name} sentences found: {len(target_sentences)}")

    if len(english_sentences) != len(target_sentences):
        print(f"⚠️  Warning: English sentences ({len(english_sentences)}) and {target_lang_name} sentences ({len(target_sentences)}) count mismatch!")

        # Try to fix by using the minimum length
        min_len = min(len(english_sentences), len(target_sentences))
        if min_len > 0:
            print(f"   Truncating to {min_len} sentences")
            english_sentences = english_sentences[:min_len]
            target_sentences = target_sentences[:min_len]
        else:
            print("❌ No sentences to save!")
            return None, None

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Save English sentences
    english_output = os.path.join(output_dir, f"english_{target_lang_name.lower()}.txt")
    with open(english_output, 'w', encoding='utf-8') as f:
        for sent in english_sentences:
            f.write(sent + '\n')

    # Save Target language sentences
    target_output = os.path.join(output_dir, f"{target_lang_name.lower()}_only.txt")
    with open(target_output, 'w', encoding='utf-8') as f:
        for sent in target_sentences:
            f.write(sent + '\n')

    # Display counts in Colab output
    print(f"\n✅ Files created successfully!")
    print(f"   📄 English file: {english_output}")
    print(f"      → {len(english_sentences)} English sentences")
    print(f"   📄 {target_lang_name} file: {target_output}")
    print(f"      → {len(target_sentences)} {target_lang_name} sentences")

    return english_output, target_output

# ============================================
# PROCESS BOTH FILES
# ============================================

# Define the files and their target languages
files_to_process = [
    {
        "input_file": "/content/translated_assamese.txt",
        "target_lang": "Assamese",
        "output_dir": "/content/drive/MyDrive"
    },
    {
        "input_file": "/content/translated_manipuri.txt",
        "target_lang": "Manipuri",
        "output_dir": "/content/drive/MyDrive"
    }
]

print("="*80)
print("🎯 SPLITTING TRANSLATED FILES INTO ENGLISH AND TARGET LANGUAGE")
print("="*80)

for file_info in files_to_process:
    input_file = file_info["input_file"]
    target_lang = file_info["target_lang"]
    output_dir = file_info["output_dir"]

    # Check if file exists
    if not os.path.exists(input_file):
        print(f"\n❌ File not found: {input_file}")
        print("📁 Please ensure the file is uploaded to /content/ directory")
        continue

    # Process the file
    english_file, target_file = split_translation_file(
        input_file,
        output_dir,
        target_lang
    )

# ============================================
# VERIFY ALL FILES
# ============================================
print("\n" + "="*80)
print("📊 SUMMARY")
print("="*80)

output_dir = "/content/drive/MyDrive"

# Check all created files
expected_files = [
    ("english_assamese.txt", "Assamese"),
    ("assamese_only.txt", "Assamese"),
    ("english_manipuri.txt", "Manipuri"),
    ("manipuri_only.txt", "Manipuri")
]

for filename, lang in expected_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            # Remove empty lines for counting
            non_empty = [line for line in lines if line.strip()]
            print(f"✅ {filename}: {len(non_empty)} sentences ({lang})")
    else:
        print(f"❌ {filename}: Not found")

print("\n" + "="*80)
print("✅ All files processed successfully!")
print(f"📁 Output directory: {output_dir}")
print("="*80)

# ============================================
# SAMPLE PREVIEW
# ============================================
def preview_file(filepath, num_lines=3, lang_name=""):
    """Display a preview of the file"""
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            non_empty = [line.strip() for line in lines if line.strip()]
            if non_empty:
                print(f"\n📝 Preview of {lang_name} ({filepath}):")
                for i, line in enumerate(non_empty[:num_lines], 1):
                    # Truncate long lines
                    display_line = line[:150] + "..." if len(line) > 150 else line
                    print(f"   {i}. {display_line}")

print("\n" + "="*80)
print("👀 FILE PREVIEWS")
print("="*80)

preview_file(
    "/content/drive/MyDrive/english_assamese.txt",
    num_lines=3,
    lang_name="English (Assamese translations)"
)
preview_file(
    "/content/drive/MyDrive/assamese_only.txt",
    num_lines=3,
    lang_name="Assamese"
)
preview_file(
    "/content/drive/MyDrive/english_manipuri.txt",
    num_lines=3,
    lang_name="English (Manipuri translations)"
)
preview_file(
    "/content/drive/MyDrive/manipuri_only.txt",
    num_lines=3,
    lang_name="Manipuri"
)

print("\n" + "="*80)
print("🎉 Done!")
print("="*80)

🎯 SPLITTING TRANSLATED FILES INTO ENGLISH AND TARGET LANGUAGE

📁 Processing: translated_assamese.txt
   Total lines in file: 3004
   English sentences found: 1000
   Assamese sentences found: 1000

✅ Files created successfully!
   📄 English file: /content/drive/MyDrive/english_assamese.txt
      → 1000 English sentences
   📄 Assamese file: /content/drive/MyDrive/assamese_only.txt
      → 1000 Assamese sentences

📁 Processing: translated_manipuri.txt
   Total lines in file: 3004
   English sentences found: 1000
   Manipuri sentences found: 1000

✅ Files created successfully!
   📄 English file: /content/drive/MyDrive/english_manipuri.txt
      → 1000 English sentences
   📄 Manipuri file: /content/drive/MyDrive/manipuri_only.txt
      → 1000 Manipuri sentences

📊 SUMMARY
✅ english_assamese.txt: 1000 sentences (Assamese)
✅ assamese_only.txt: 1000 sentences (Assamese)
✅ english_manipuri.txt: 1000 sentences (Manipuri)
✅ manipuri_only.txt: 1000 sentences (Manipuri)

✅ All files processed succ